Importation des bibliothèques

In [1]:
import os
import random
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import copy #pour copier base_model
import numpy as np
import pandas as pd
import itertools
import csv
import pickle

In [2]:
from models import SimpleCNN, CNN_MCdropout
from dataset import load_cifar10
from train import train_model, evaluate
from utils import mc_predict_mean_probs, generate_mc_outputs
from accuracy import accuracy_threshold, monotonic_rearrangement, isotonic_regression, monotonicity_penalty

In [3]:
print(os.getcwd())  # donne le répertoire courant

c:\Users\Invite\Documents\INRIA\MCDropout


Fixation du seed pour la reproductibilité

In [4]:
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [5]:
trainloader, valloader, testloader, classes = load_cifar10(batch_size=128, val_ratio=0.1)

In [6]:
device = torch.device("cuda")
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0)) #c'est censé être RTX

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4060


In [7]:
save_path = "best.pt"

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

Chargement du modèle sauvegardé


Test manuel de plusieurs dico_layers et stockage des résultats

In [8]:
layer_names = ["conv1", "conv2", "conv3", "fc1"]
dropout_values = [round(x, 1) for x in np.arange(0.1, 1, 0.1)]

grid_configs = []
for r in range(1, len(layer_names)+1):
    for layers in itertools.combinations(layer_names, r):
        for probs in itertools.product(dropout_values, repeat=len(layers)):
            dico = {layer: p for layer, p in zip(layers, probs)}
            for before in [True, False]:
                grid_configs.append({
                    "dico_layers": dico,
                    "before": before
                })

print(f"Nombre total de configurations : {len(grid_configs)}")

Nombre total de configurations : 19998


Grid search sur dico_layers et stockage des résultats

In [9]:
all_results = [] 
all_accuracy_curves = []

metrics_list = [
    "variance_predicted",
    "variance_max",
    "predictive_entropy_predicted",
    "predictive_entropy_max"
]

min_penalties = {metric: {"penalty_isotone": float("inf"), "penalty_rearrangement": float("inf"), "config_ids_iso": [], "config_ids_rearr": []} for metric in metrics_list}

for i, config in enumerate(grid_configs):
    dico_layers = config["dico_layers"]
    before = config["before"]

    model_mc = CNN_MCdropout(copy.deepcopy(base_model), dico_layers=dico_layers, before=before).to(device)
    
    all_probs = []
    all_Y = []
    all_metric_values = {metric: [] for metric in metrics_list}
    all_mean_probs = []
    
    for X, Y in testloader:
        X, Y = X.to(device), Y.to(device)
        probs, _ = mc_predict_mean_probs(model_mc, X, T=100, verbose=False)
        all_probs.append(probs)
        all_Y.append(Y)
        outputs, mean_probs, metric_values, _, _ = generate_mc_outputs(
            model_mc, X, T=100, metrics=metrics_list, labels=Y, verbose=False
        )
        all_mean_probs.append(mean_probs)
        for metric in metrics_list:
            all_metric_values[metric].append(metric_values[metric])
    
    probs = torch.cat(all_probs)
    Y = torch.cat(all_Y)
    Y_hat = probs.argmax(1)
    acc = (Y_hat == Y).float().mean().item()
    mean_probs = torch.cat(all_mean_probs).mean()
    metric_values = {metric: torch.cat(all_metric_values[metric]) for metric in metrics_list}

    # Courbes d'accuracy et calcul des pénalités
    penalties = {}
    accuracy_curves = {}
    for metric, color in zip([
        "variance_predicted", "variance_max",
        "predictive_entropy_predicted", "predictive_entropy_max"
    ], ["royalblue", "seagreen", "deeppink", "darkorchid"]):
        quantiles, thresholds, accuracies = accuracy_threshold(
            Y_hat, Y, metric_values[metric], metric_name=metric, color=color, display=False
        )
        accuracy_curves[metric] = {
            "quantiles": quantiles,
            "thresholds": thresholds,
            "accuracies": accuracies
        }
        penalty_iso = monotonicity_penalty(quantiles, accuracies, method="isotonic")
        penalty_rearr = monotonicity_penalty(quantiles, accuracies, method="rearrangement")
        penalties[metric] = {
            "penalty_isotone": penalty_iso,
            "penalty_rearrangement": penalty_rearr
        }

        if penalty_iso < min_penalties[metric]["penalty_isotone"]:
            min_penalties[metric]["penalty_isotone"] = penalty_iso
            min_penalties[metric]["config_ids_iso"] = [i]
        elif penalty_iso == min_penalties[metric]["penalty_isotone"]:
            min_penalties[metric]["config_ids_iso"].append(i)

        if penalty_rearr < min_penalties[metric]["penalty_rearrangement"]:
            min_penalties[metric]["penalty_rearrangement"] = penalty_rearr
            min_penalties[metric]["config_ids_rearr"] = [i]
        elif penalty_rearr == min_penalties[metric]["penalty_rearrangement"]:
            min_penalties[metric]["config_ids_rearr"].append(i)

    result = {
        "config_id": i,
        "dico_layers": dico_layers,
        "before": before,
        "acc": acc,
        "mc_mean_probs": mean_probs.item(), 
        **{metric: metric_values[metric].mean().item() if isinstance(metric_values[metric], torch.Tensor) else metric_values[metric] for metric in metrics_list},
        "penalties": penalties
    }
    all_results.append(result)
    all_accuracy_curves.append({
        "config_id": i,
        "accuracy_curves": accuracy_curves
    })

    if (i+1) % 10 == 0 or i == len(grid_configs)-1:
        print(f"Config {i+1}/{len(grid_configs)} traitée.")

# Sauvegarde au format pickle
with open("all_results.pkl", "wb") as f:
    pickle.dump(all_results, f)
print("Résultats sauvegardés dans all_results.pkl")

# Sauvegarde également en CSV
# Correction : convertir les valeurs du dico_layers en float natif pour éviter np.float64 dans le CSV
def dico_layers_to_str(dico):
    # Convertit toutes les valeurs en float natif pour une sérialisation propre
    return str({k: float(v) for k, v in dico.items()})

df_results = pd.DataFrame(all_results)
df_results['dico_layers'] = df_results['dico_layers'].apply(dico_layers_to_str)
df_results.to_csv("all_results.csv", index=False)
print("Résultats sauvegardés dans all_results.csv")

for metric in metrics_list:
    print(f"Métrique '{metric}' :")
    print(f"  - Pénalité isotone minimisée pour config(s) {min_penalties[metric]['config_ids_iso']} avec valeur {min_penalties[metric]['penalty_isotone']}")
    print(f"  - Pénalité rearrangement minimisée pour config(s) {min_penalties[metric]['config_ids_rearr']} avec valeur {min_penalties[metric]['penalty_rearrangement']}")


Config 10/19998 traitée.
Config 20/19998 traitée.
Config 30/19998 traitée.
Config 40/19998 traitée.
Config 50/19998 traitée.
Config 60/19998 traitée.
Config 70/19998 traitée.
Config 80/19998 traitée.
Config 90/19998 traitée.
Config 100/19998 traitée.
Config 110/19998 traitée.
Config 120/19998 traitée.
Config 130/19998 traitée.
Config 140/19998 traitée.
Config 150/19998 traitée.
Config 160/19998 traitée.
Config 170/19998 traitée.
Config 180/19998 traitée.
Config 190/19998 traitée.
Config 200/19998 traitée.
Config 210/19998 traitée.
Config 220/19998 traitée.
Config 230/19998 traitée.
Config 240/19998 traitée.
Config 250/19998 traitée.
Config 260/19998 traitée.
Config 270/19998 traitée.
Config 280/19998 traitée.
Config 290/19998 traitée.
Config 300/19998 traitée.
Config 310/19998 traitée.
Config 320/19998 traitée.
Config 330/19998 traitée.
Config 340/19998 traitée.
Config 350/19998 traitée.
Config 360/19998 traitée.
Config 370/19998 traitée.
Config 380/19998 traitée.
Config 390/19998 trai

In [ ]:
# rajouter affichage tableau avec double index comme pour l'autre notebook
# rajouter 3 colonnes pour la config : layer, proba, before pour faire une recherche (ex: before = true)
import ast

df_results = pd.read_csv("all_results.csv")

# Nettoyage de la colonne penalties pour enlever np.float64(...)
def clean_penalties_str(s):
    # Remplace np.float64(123.4) par 123.4
    import re
    return re.sub(r'np\.float64\(([^)]+)\)', r'\1', s)

df_results['penalties'] = df_results['penalties'].apply(clean_penalties_str)

# Extraction des infos de config pour les colonnes d'index
def extract_layers(dico_layers_str):
    try:
        dico = ast.literal_eval(dico_layers_str)
        if isinstance(dico, dict):
            return ','.join(sorted(dico.keys()))
        else:
            return ''
    except Exception as e:
        print(f"[WARN] Problème parsing dico_layers: {dico_layers_str} -- {e}")
        return ''

def extract_proba(dico_layers_str):
    try:
        dico = ast.literal_eval(dico_layers_str)
        if isinstance(dico, dict) and dico:
            return list(dico.values())[0]
        else:
            return None
    except Exception as e:
        print(f"[WARN] Problème parsing dico_layers: {dico_layers_str} -- {e}")
        return None

df_results['layers'] = df_results['dico_layers'].apply(extract_layers)
df_results['proba'] = df_results['dico_layers'].apply(extract_proba)
df_results['before'] = df_results['before'].astype(bool)

# Extraction des pénalités pour chaque métrique
metrics = [
    "variance_predicted",
    "variance_max",
    "predictive_entropy_predicted",
    "predictive_entropy_max"
]
penalties = ["penalty_isotone", "penalty_rearrangement"]

def extract_penalty(d, metric, penalty):
    try:
        dico = ast.literal_eval(d)
        return dico[metric][penalty]
    except Exception as e:
        print(f"[WARN] Problème parsing penalties: {d} -- {e}")
        return None

for metric in metrics:
    for penalty in penalties:
        col = (metric, penalty)
        # Supprime la colonne si elle existe déjà pour éviter les doublons
        if col in df_results.columns:
            df_results = df_results.drop(columns=[col])
        df_results[col] = df_results['penalties'].apply(
            lambda d: extract_penalty(d, metric, penalty)
        )

# Création du MultiIndex pour les colonnes
columns = pd.MultiIndex.from_product([metrics, penalties])

# Création du MultiIndex pour les lignes
df_results = df_results.reset_index(drop=True)  # retire tout index existant pour éviter les doublons

# Supprime les doublons sur ['layers', 'proba', 'before'] pour garantir un index unique
df_results = df_results.drop_duplicates(subset=['layers', 'proba', 'before'])

df_results.set_index(['layers', 'proba', 'before'], inplace=True)

# Vérification unicité index et colonnes
assert df_results.index.is_unique, "Index non unique"
assert df_results.columns.is_unique, "Colonnes non uniques"

# Trouver les valeurs minimales pour chaque métrique/penalty
min_vals = {}
for metric in metrics:
    for penalty in penalties:
        col = (metric, penalty)
        min_vals[col] = df_results[col].min()

# Fonction de surlignage
def highlight_min(s):
    is_min = s == min_vals[s.name]
    return ['background-color: #228B22' if v else '' for v in is_min]

# Pour l'affichage, si plusieurs couches ont des probas différentes, il faut les afficher explicitement.
# On ajoute une colonne 'probas_str' qui liste toutes les probas par couche.

def extract_probas_str(dico_layers_str):
    try:
        dico = ast.literal_eval(dico_layers_str)
        if isinstance(dico, dict) and dico:
            return ','.join([f"{k}:{v}" for k, v in sorted(dico.items())])
        else:
            return ''
    except Exception as e:
        print(f"[WARN] Problème parsing dico_layers: {dico_layers_str} -- {e}")
        return ''

df_results['probas_str'] = df_results['dico_layers'].apply(extract_probas_str)

# Supprime les doublons sur ['layers', 'probas_str', 'before'] pour garantir un index unique
df_results = df_results.drop_duplicates(subset=['layers', 'probas_str', 'before'])

df_results.set_index(['layers', 'probas_str', 'before'], inplace=True)

# Vérification unicité index et colonnes
assert df_results.index.is_unique, "Index non unique"
assert df_results.columns.is_unique, "Colonnes non uniques"

# Affichage du tableau formaté avec surlignage
display(df_results[columns].sort_index().style.apply(highlight_min, subset=columns))